# Finetune Nanochat 0.7B — Darija Chat SFT

**Model:** `KandirResearch/Nanochat-Moroccan-Instruct-0.7B` (702M params, 32K vocab, 2048 ctx)

**Dataset:** `Lyte/darija-translation-data-large-v0.1` (~35K rows)
- **opus** (~1.5K) — reasoning with thinking chains (problem → thinking → solution)
- **alpaca** (~33K) — instruction-following conversations
- **smoltalk** (182) — chat conversations

**Strategy:** Darija-native chat SFT — all conversations in Darija
1. Alpaca/SmolTalk — user asks in Darija, model responds in Darija
2. Opus — chain-of-thought reasoning in Darija

**Output:** `Lyte/nanochat-moroccan-instruct-v2`

In [ ]:
!pip install -q trl datasets huggingface_hub peft
!pip install -U bitsandbytes>=0.46.1

In [ ]:
import os
HF_TOKEN = os.environ["HF_TOKEN"]
!huggingface-cli login --token $HF_TOKEN

In [ ]:
import os
import json
import random
import torch

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # single GPU — avoid DataParallel

from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# ============================================================
# Config
# ============================================================

MODEL_ID = "KandirResearch/Nanochat-Moroccan-Instruct-0.7B"
DATASET_REPO = "Lyte/darija-translation-data-large-v0.1"
OUTPUT_HF_REPO = "Lyte/nanochat-moroccan-instruct-v2"

SEED = 42
VAL_RATIO = 0.05

random.seed(SEED)
torch.manual_seed(SEED)

print(f"Model:   {MODEL_ID}")
print(f"Dataset: {DATASET_REPO}")
print(f"Output:  {OUTPUT_HF_REPO}")
print(f"GPU:     {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"VRAM:    {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

In [ ]:
# ============================================================
# System prompt (Darija reasoning assistant)
# ============================================================

SYS_REASONING = (
    "أنت مساعد ذكي كيجاوب بالدارجة المغربية. فكّر مزيان قبل ما تجاوب."
)

In [ ]:
# ============================================================
# Load all data configs
# ============================================================

print("Loading datasets...")

opus_ds = load_dataset(DATASET_REPO, "opus", split="train", token=HF_TOKEN)
alpaca_ds = load_dataset(DATASET_REPO, "alpaca", split="train", token=HF_TOKEN)
smoltalk_ds = load_dataset(DATASET_REPO, "smoltalk", split="train", token=HF_TOKEN)

print(f"  opus:     {len(opus_ds):,} rows")
print(f"  alpaca:   {len(alpaca_ds):,} rows")
print(f"  smoltalk: {len(smoltalk_ds):,} rows")
print(f"  Total:    {len(opus_ds) + len(alpaca_ds) + len(smoltalk_ds):,} rows")

In [ ]:
# ============================================================
# Clean degenerate translations
# ============================================================
# Filters out rows where the translation model fell into
# repetition loops or produced excessive whitespace.

import re

def is_degenerate(text: str) -> bool:
    """Return True if text shows signs of degenerate generation."""
    if not text:
        return False

    # 1. Phrase repeated 5+ times (4+ words, len > 15)
    words = text.split()
    for n in range(4, 9):
        if len(words) < n * 5:
            continue
        seen = {}
        for i in range(len(words) - n + 1):
            chunk = " ".join(words[i:i+n])
            if len(chunk) > 15:
                seen[chunk] = seen.get(chunk, 0) + 1
                if seen[chunk] >= 5:
                    return True

    # 2. Excessive consecutive newlines (8+)
    if re.search(r"\n{8,}", text):
        return True

    # 3. Single character repeated 20+ times (not space)
    if re.search(r"([^\s])\1{19,}", text):
        return True

    return False


def row_is_clean(row, fields):
    """Check all specified fields of a row for degeneration."""
    for f in fields:
        val = row.get(f, "")
        if isinstance(val, list):
            for msg in val:
                if isinstance(msg, dict) and is_degenerate(msg.get("content", "")):
                    return False
        elif isinstance(val, str) and is_degenerate(val):
            return False
    return True


# --- Filter each dataset ---
opus_before = len(opus_ds)
alpaca_before = len(alpaca_ds)
smoltalk_before = len(smoltalk_ds)

opus_fields = ["problem", "thinking", "solution"]
conv_fields = ["messages"]

opus_ds = opus_ds.filter(lambda row: row_is_clean(row, opus_fields))
alpaca_ds = alpaca_ds.filter(lambda row: row_is_clean(row, conv_fields))
smoltalk_ds = smoltalk_ds.filter(lambda row: row_is_clean(row, conv_fields))

print(f"Cleaned degenerate rows:")
print(f"  opus:     {opus_before:,} → {len(opus_ds):,}  (removed {opus_before - len(opus_ds):,})")
print(f"  alpaca:   {alpaca_before:,} → {len(alpaca_ds):,}  (removed {alpaca_before - len(alpaca_ds):,})")
print(f"  smoltalk: {smoltalk_before:,} → {len(smoltalk_ds):,}  (removed {smoltalk_before - len(smoltalk_ds):,})")
print(f"  Total remaining: {len(opus_ds) + len(alpaca_ds) + len(smoltalk_ds):,}")

In [ ]:
# ============================================================
# Build SFT rows — Darija chat only
# ============================================================

def make_row(prompt_msgs, completion_msgs):
    """Make a TRL-compatible SFT row with prompt + completion."""
    return {"prompt": prompt_msgs, "completion": completion_msgs}


def build_conv_row(da_msgs):
    """Build a single Darija chat row from a conversation.

    prompt  = all messages except the last
    completion = the last assistant message
    """
    if len(da_msgs) < 2:
        return None
    prompt = [{"role": m["role"], "content": m["content"]} for m in da_msgs[:-1]]
    completion = [{"role": da_msgs[-1]["role"], "content": da_msgs[-1]["content"]}]
    return make_row(prompt, completion)


def build_opus_row(row):
    """Build a Darija reasoning chat row from an Opus record.

    problem → thinking + solution (chain-of-thought in Darija)
    """
    problem_da = row["problem"]
    thinking_da = row.get("thinking", "")
    solution_da = row["solution"]

    if not problem_da or not solution_da:
        return None

    answer = solution_da
    if thinking_da:
        answer = thinking_da + "\n\n" + solution_da

    prompt = [
        {"role": "system", "content": SYS_REASONING},
        {"role": "user", "content": problem_da},
    ]
    completion = [{"role": "assistant", "content": answer}]
    return make_row(prompt, completion)


# ============================================================
# Process all records
# ============================================================

all_train_rows = []
all_val_rows = []

# --- Alpaca + SmolTalk (conversation format) ---
conv_records = []
for ds, source in [(alpaca_ds, "alpaca"), (smoltalk_ds, "smoltalk")]:
    for row in ds:
        conv_records.append(row["messages"])

random.shuffle(conv_records)
val_cut = int(len(conv_records) * VAL_RATIO)

for i, da_msgs in enumerate(conv_records):
    sft_row = build_conv_row(da_msgs)
    if sft_row is None:
        continue
    if i < val_cut:
        all_val_rows.append(sft_row)
    else:
        all_train_rows.append(sft_row)

# --- Opus (reasoning format) ---
opus_records = list(opus_ds)
random.shuffle(opus_records)
val_cut_opus = int(len(opus_records) * VAL_RATIO)

for i, row in enumerate(opus_records):
    sft_row = build_opus_row(row)
    if sft_row is None:
        continue
    if i < val_cut_opus:
        all_val_rows.append(sft_row)
    else:
        all_train_rows.append(sft_row)

random.shuffle(all_train_rows)

train_ds = Dataset.from_list(all_train_rows)
val_ds = Dataset.from_list(all_val_rows)

print(f"Train rows: {len(train_ds):,}")
print(f"Val rows:   {len(val_ds):,}")
print(f"  Conv records: {len(conv_records):,} (alpaca + smoltalk)")
print(f"  Opus records: {len(opus_records):,}")

In [ ]:
# ============================================================
# Token length analysis (on actual SFT rows)
# ============================================================

from transformers import AutoTokenizer as _AT

_tok = _AT.from_pretrained(MODEL_ID, trust_remote_code=True, token=HF_TOKEN)

def measure_token_len(row):
    msgs = row["prompt"] + row["completion"]
    text = _tok.apply_chat_template(msgs, tokenize=False)
    return len(_tok.encode(text))

print("Measuring token lengths on train set...")
lengths = [measure_token_len(train_ds[i]) for i in range(len(train_ds))]
lengths.sort()

n = len(lengths)
avg = sum(lengths) / n
median = lengths[n // 2]
p90 = lengths[int(n * 0.90)]
p95 = lengths[int(n * 0.95)]
p98 = lengths[int(n * 0.98)]
p99 = lengths[int(n * 0.99)]
mx = lengths[-1]

over_512 = sum(1 for l in lengths if l > 512)
over_1024 = sum(1 for l in lengths if l > 1024)
over_2048 = sum(1 for l in lengths if l > 2048)

print(f"\nTrain set: {n:,} rows")
print(f"  avg={avg:.0f}  median={median}  max={mx}")
print(f"  p90={p90}  p95={p95}  p98={p98}  p99={p99}")
print(f"  >512: {over_512:,} ({100*over_512/n:.1f}%)")
print(f"  >1024: {over_1024:,} ({100*over_1024/n:.1f}%)")
print(f"  >2048: {over_2048:,} ({100*over_2048/n:.1f}%)")
print(f"\n→ Suggested max_length based on p99: {min(2048, max(256, (p99 // 128 + 1) * 128))}")

del _tok

In [ ]:
# ============================================================
# Inspect training examples
# ============================================================

for i in range(5):
    row = train_ds[i]
    print(f"\n{'='*60}")
    print(f"Example {i+1}")
    print(f"{'='*60}")
    for m in row["prompt"]:
        role = m["role"]
        text = m["content"][:120]
        suffix = "..." if len(m["content"]) > 120 else ""
        print(f"  [{role}]: {text}{suffix}")
    for m in row["completion"]:
        role = m["role"]
        text = m["content"][:120]
        suffix = "..." if len(m["content"]) > 120 else ""
        print(f"  [{role}]: {text}{suffix}")

In [ ]:
# ============================================================
# Load tokenizer
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, trust_remote_code=True, token=HF_TOKEN
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer: {tokenizer.name_or_path}")
print(f"Vocab size: {tokenizer.vocab_size:,}")
print(f"EOS: {tokenizer.eos_token} (id={tokenizer.eos_token_id})")
print(f"Pad: {tokenizer.pad_token}")

# Verify chat template works
test_msgs = [
    {"role": "user", "content": "كيفاش نطيب الطاجين؟"},
]
print(f"\nChat template test:")
print(tokenizer.apply_chat_template(test_msgs, tokenize=False, add_generation_prompt=True))

In [ ]:
# ============================================================
# QLoRA 4-bit config
# ============================================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    trust_remote_code=True,
    device_map={"":0},
    attn_implementation="eager",
    token=HF_TOKEN,
)

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)
model.config.use_cache = False

# LoRA targets: all attention + MLP projections
# Nanochat uses c_q/c_k/c_v/c_proj (attn) and c_fc/c_proj (MLP)
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules=["c_q", "c_k", "c_v", "c_fc", "c_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model loaded: {MODEL_ID}")
print(f"Total params:     {total_params:,}")
print(f"LoRA r={peft_config.r}, alpha={peft_config.lora_alpha}")

In [ ]:
# ============================================================
# Training config
# ============================================================
#
# 0.7B model with 32K vocab is much lighter than the 3.35B/256K aya.
# Adjust batch size based on available VRAM:
#   - RTX 4050 (6GB):  bs=1, grad_accum=16
#   - RTX 3090 (24GB): bs=8, grad_accum=4
#   - A100 (40/80GB):  bs=16, grad_accum=2

vram_gb = torch.cuda.get_device_properties(
    0).total_memory / 1e9 if torch.cuda.is_available() else 6

if vram_gb >= 40:
    BS, GA = 16, 2
elif vram_gb >= 20:
    BS, GA = 8, 4
elif vram_gb >= 10:
    BS, GA = 4, 8
else:
    BS, GA = 1, 16

print(f"VRAM: {vram_gb:.1f} GB → batch_size={BS}, grad_accum={GA}, eff_batch={BS*GA}")

training_args = SFTConfig(
    output_dir="nanochat-darija-v2",
    num_train_epochs=2,
    per_device_train_batch_size=BS,
    per_device_eval_batch_size=BS,
    gradient_accumulation_steps=GA,
    learning_rate=3e-5,
    weight_decay=0.05,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=True,
    logging_steps=10,
    eval_steps=250,
    eval_strategy="steps",
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    optim="paged_adamw_8bit",
    gradient_checkpointing=False,
    gradient_checkpointing_kwargs={},
    max_grad_norm=0.3,
    report_to="none",
    seed=SEED,
    max_length=1024,
    packing=False,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
)

In [ ]:
# ============================================================
# Trainer
# ============================================================

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    peft_config=peft_config,
)

print(f"\n{'='*70}")
print("Nanochat 0.7B — Darija SFT v2")
print(f"{'='*70}")
print(f"Base model:       {MODEL_ID}")
print(f"Train rows:       {len(train_ds):,}")
print(f"Val rows:         {len(val_ds):,}")
print(f"Epochs:           {training_args.num_train_epochs}")
print(f"LoRA r:           {peft_config.r}")
print(f"Max length:       {training_args.max_length}")
print(f"LR:               {training_args.learning_rate}")
print(f"Eff batch:        {BS} x {GA} = {BS * GA}")
print(f"{'='*70}\n")

In [ ]:
trainer.train()

In [ ]:
# ============================================================
# Merge LoRA + Push to HF
# ============================================================

from peft import PeftModel
import glob

# Find best checkpoint
checkpoints = sorted(glob.glob("nanochat-darija-v2/checkpoint-*"))
best_ckpt = checkpoints[-1] if checkpoints else "nanochat-darija-v2"
print(f"Using checkpoint: {best_ckpt}")

# Load base model (full precision for merge)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map={"":0},
    trust_remote_code=True,
    attn_implementation="eager",
    token=HF_TOKEN,
)

# Load LoRA adapter
model_with_lora = PeftModel.from_pretrained(base_model, best_ckpt)

# Merge
print("Merging LoRA adapter...")
merged_model = model_with_lora.merge_and_unload()

# Save locally
print("Saving locally...")
merged_model.save_pretrained("nanochat-darija-v2-merged", safe_serialization=True)
tokenizer.save_pretrained("nanochat-darija-v2-merged")

# Push to HF
print(f"Pushing to {OUTPUT_HF_REPO}...")
merged_model.push_to_hub(OUTPUT_HF_REPO, private=True, token=HF_TOKEN)
tokenizer.push_to_hub(OUTPUT_HF_REPO, private=True, token=HF_TOKEN)

print(f"\nPushed to https://huggingface.co/{OUTPUT_HF_REPO}")

In [ ]:
# ============================================================
# Quick test
# ============================================================

merged_model.eval()

darija_prompts = [
    "كيفاش نطيب الطاجين؟",
    "شنو هي عاصمة المغرب؟",
    "عطيني 3 نصائح باش نتعلم لغة جديدة.",
    "شرح ليا شنو هو machine learning بالدارجة.",
]

for p in darija_prompts:
    msgs = [{"role": "user", "content": p}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(merged_model.device)
    prompt_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        out = merged_model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.6,
            top_k=200,
            top_p=0.85,
            min_p=0.01,
            repetition_penalty=1.1,
            do_sample=True,
            use_cache=False,
        )
    response = tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True)
    print(f"\nQ: {p}")
    print(f"A: {response[:300]}")
    print("-" * 60)